In [1]:
import numpy as np
import os
from os.path import join
import re
import matplotlib.lines as mlines
from matplotlib.ticker import FormatStrFormatter
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from matplotlib import cm
import matplotlib.pyplot as plt
from rethinking_neural_id.paths import RepoPaths

In [2]:
paths = RepoPaths.default()
paths.cnn_figs_root.mkdir(parents=True, exist_ok=True)
results_folder = str(paths.cnn_metrics_root)

In [3]:
# Architectures considered in the CNN analysis
architectures = ['alexnet', 'vgg16', 'resnet18', 'resnet34'] 

# Load all metrics for all architectures
ID = {}    
IDerr = {}
names = {}
cossim = {}
cossimerr = {}
L2_dist = {}
L2_disterr = {}
Entropy = {}
Entropyerr = {}
kNN_dist = {}
kNN_disterr = {}
Depths = {}
ambdims = {}
for arch in architectures:
    ID[arch] = np.load(join(results_folder, arch + '_ID.npy')) 
    IDerr[arch] = np.load(join(results_folder, arch + '_IDerr.npy')) 
    cossim[arch] = np.load(join(results_folder, arch + '_COSSIM.npy'))
    cossimerr[arch] = np.load(join(results_folder, arch + '_COSSIMerr.npy'))
    L2_dist[arch] = np.load(join(results_folder, arch + '_L2_DIST.npy'))
    L2_disterr[arch] = np.load(join(results_folder, arch + '_L2_DISTerr.npy'))
    Entropy[arch] = np.load(join(results_folder, arch + '_ENTROPY.npy'))
    Entropyerr[arch] = np.load(join(results_folder, arch + '_ENTROPYerr.npy'))
    kNN_dist[arch] = np.load(join(results_folder, arch + '_KNN_DIST.npy'))
    kNN_disterr[arch] = np.load(join(results_folder, arch + '_KNN_DISTerr.npy'))
    Depths[arch] = np.load(join(results_folder, arch + '_depths.npy'))
    ambdims[arch] = np.load(join(results_folder, arch + '_ambdims.npy'))
    names[arch] = np.load(join(results_folder, arch + '_names.npy'), allow_pickle=True)


### Class-specific ID estimates

In [ ]:
architectures = ['alexnet', 'vgg16', 'resnet18', 'resnet34'] 
category_names = ['koalas', 'shih-tzu', 'rhodesian', 'yorkshire',
                  'vizsla', 'setter', 'butterfly']

# Plot styling
lw = 2
alpha_line = 0.9
alpha_ci = 0.1   
ms = 8
fs = 18

def pretty_title(a):
    a = a.lower().replace('_','-')
    if a.startswith('alex'): return 'AlexNet'
    if a.startswith('vgg'): return 'VGG-' + ''.join(ch for ch in a if ch.isdigit())
    if a.startswith('resnet'): return 'ResNet-' + ''.join(ch for ch in a if ch.isdigit())
    return a

# Consistent colors for each class across all subplots
base_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
color_map = {cname: base_colors[i % len(base_colors)] for i, cname in enumerate(category_names)}

# Make figure with 4 panels, shared y-axis
fig, axs = plt.subplots(1, len(architectures), figsize=(22, 5.5), sharey=True)
if len(architectures) == 1:
    axs = [axs]  # handle edge case if list gets shortened

for ax, arch in zip(axs, architectures):
    # Determine relative depths per model
    n_layers = ID[arch].shape[1]
    try:
        rdepths = np.asarray(Depths[arch], dtype=float)
        assert rdepths.shape[0] == n_layers
        rdepths = rdepths / rdepths[-1]
    except Exception:
        rdepths = np.linspace(0.0, 1.0, n_layers)

    # Plot each class with consistent color
    for j, cname in enumerate(category_names):
        y = ID[arch][j, :]
        yerr = IDerr[arch][j, :]

        # Mean line
        ax.plot(
            rdepths, y, '-o',
            linewidth=lw, markersize=ms, alpha=alpha_line,
            label=cname, color=color_map[cname]
        )
        # Shaded CI
        ax.fill_between(
            rdepths, y - yerr, y + yerr,
            color=color_map[cname], alpha=alpha_ci
        )

    ax.set_title(arch, fontsize=fs)          
    ax.set_xlim(-0.05, 1.05)
    ax.grid(axis='y', alpha=0.3)
    ax.grid(axis='x', alpha=0.3)
    ax.set_xticks(np.linspace(0, 1, 6))
    ax.tick_params(labelsize=fs - 2)
    ax.set_title(pretty_title(arch), fontsize=fs+12, fontweight='bold', pad=17)

# Common labels (y on the shared axis, x for the whole row)
axs[0].set_ylabel('Estimated ID', fontsize=fs)
for ax in axs:
    ax.set_xlabel('Relative Depth', fontsize=fs)

# Build a single legend on the far right
legend_handles = [
    Line2D([0], [0], marker='o', linestyle='-',
           linewidth=lw, markersize=ms,
           color=color_map[c], label=c)
    for c in category_names
]
plt.subplots_adjust(right=0.84, wspace=0.18)
fig.legend(
    handles=legend_handles, labels=category_names,
    fontsize=fs - 2, loc='center left', bbox_to_anchor=(0.865, 0.5),
    frameon=False
)

# Tight layout after adjusting space for legend
fig.tight_layout(rect=[0, 0, 0.87, 1])

# Save figure
save_name = "id_classes_all_models.pdf"
out_path = paths.cnn_figs_root / save_name
fig.savefig(out_path, bbox_inches='tight')
print(f"Saved figure as {save_name}")

plt.show()

In [ ]:
arch = 'resnet34'
category_names = ['koalas', 'shih-tzu', 'rhodesian', 'yorkshire',
                  'vizsla', 'setter', 'butterfly']

n_layers = ID[arch].shape[1]
try:
    rdepths = np.asarray(Depths[arch], dtype=float)
    assert rdepths.shape[0] == n_layers
    rdepths = rdepths / rdepths[-1]
except Exception:
    rdepths = np.linspace(0.0, 1.0, n_layers)

# plot style
lw = 2
alpha_line = 0.9
alpha_ci = 0.1 
ms = 8
fs = 18

fig, ax = plt.subplots(figsize=(10, 5.5))

# Only first 7 rows = 7 classes
for j, cname in enumerate(category_names):
    y = ID[arch][j, :]
    yerr = IDerr[arch][j, :]

    # Plot mean line
    line = ax.plot(rdepths, y, '-o', linewidth=lw, markersize=ms,
                   alpha=alpha_line, label=cname)
    color = line[0].get_color()

    # Shaded CI
    ax.fill_between(rdepths, y - yerr, y + yerr,
                    color=color, alpha=alpha_ci)

ax.set_xlabel('Relative Depth', fontsize=fs)
ax.set_ylabel('Estimated ID', fontsize=fs)
ax.set_xlim(-0.05, 1.05)
ax.grid(axis='y', alpha=0.3)
ax.grid(axis='x', alpha=0.3)
ax.set_xticks(np.linspace(0, 1, 6))
ax.tick_params(labelsize=fs - 2)

ax.legend(fontsize=fs - 2, loc='center left', bbox_to_anchor=(1.02, 0.5))
fig.tight_layout()

# Save figure
save_name = f"id_classes_{arch}.pdf"
out_path = paths.cnn_figs_root / save_name
fig.savefig(out_path, bbox_inches='tight')
print(f"Saved figure as {save_name}")

plt.show()

### NN-distances (CNNs)

In [ ]:
architectures = ['alexnet', 'vgg16', 'resnet18', 'resnet34'] 
k_values = [1, 2, 3, 4, 5]
fs = 18
lw = 2
alpha_line = 0.9
alpha_ci = 0.1 
ms = 8

def pretty_title(a):
    a = a.lower().replace('_','-')
    if a.startswith('alex'): return 'AlexNet'
    if a.startswith('vgg'): return 'VGG-' + ''.join(ch for ch in a if ch.isdigit())
    if a.startswith('resnet'): return 'ResNet-' + ''.join(ch for ch in a if ch.isdigit())
    return a

# colors
cmap = cm.get_cmap('Blues')
k_norm = np.linspace(0.45, 0.85, len(k_values))
k_to_color = {k: cmap(k_norm[i]) for i, k in enumerate(k_values)}

def stack_model_nn(model_data):
    """
    model_data: list over datasets; each item is list over layers; each item is len-5 sequence
    returns: np.ndarray of shape (num_datasets, num_layers, 5)
    """
    arrs = []
    for ds in model_data:
        a = np.asarray(ds, dtype=float) 
        if a.ndim != 2 or a.shape[1] != 5:
            raise ValueError(f"Each dataset entry must be (num_layers, 5); got {a.shape}")
        arrs.append(a)
    return np.stack(arrs, axis=0) 

fig, axs = plt.subplots(1, len(architectures), figsize=(22, 5.5))
if len(architectures) == 1:
    axs = [axs]

for ax, arch in zip(axs, architectures):
    data = stack_model_nn(kNN_dist[arch])
    num_datasets, n_layers, num_k = data.shape
    assert num_k == 5, "Expected 5 k-NN distances per layer"

    mean_over_ds = data.mean(axis=0)           
    std_over_ds  = data.std(axis=0, ddof=0)    

    # relative depths from Depths[arch] if available, else linspace
    try:
        rdepths = np.asarray(Depths[arch], dtype=float)
        assert rdepths.shape[0] == n_layers
        rdepths = rdepths / rdepths[-1]
    except Exception:
        rdepths = np.linspace(0.0, 1.0, n_layers)

    # plot each k with mean line and shaded std band
    for k_idx, k in enumerate(k_values):  
        y = mean_over_ds[:, k_idx]
        ystd = std_over_ds[:, k_idx]
        color = k_to_color[k]

        ax.plot(
            rdepths, y, '-o',
            linewidth=lw, markersize=ms, alpha=alpha_line,
            color=color, label=f'{k}-NN'
        )
        ax.fill_between(
            rdepths, y - 2*ystd, y + 2*ystd,
            color=color, alpha=alpha_ci
        )

    # axes cosmetics
    ax.grid(axis='y', alpha=0.3)
    ax.grid(axis='x', alpha=0.3)
    ax.set_xlim(-0.05, 1.05)
    ax.set_xticks(np.linspace(0, 1, 6))
    ax.tick_params(labelsize=fs - 2)
    ax.set_xlabel('Relative Depth', fontsize=fs)
    ax.set_title(pretty_title(arch), fontsize=fs+12, fontweight='bold', pad=17)

axs[0].set_ylabel('Average k-NN distance', fontsize=fs)

# Build a single legend 
legend_handles = [
    Line2D([0], [0], marker='o', linestyle='-',
           linewidth=lw, markersize=ms, color=k_to_color[k], label=f'{k}-NN')
    for k in k_values
]
plt.subplots_adjust(right=0.84, wspace=0.18)
fig.legend(
    handles=legend_handles, labels=[f'{k}-NN' for k in k_values],
    fontsize=fs - 2, loc='center left', bbox_to_anchor=(0.865, 0.5), frameon=False
)

fig.tight_layout(rect=[0, 0, 0.87, 1])

# Save figure
save_name = "nn_dists_all_models.pdf"
out_path = paths.cnn_figs_root / save_name
fig.savefig(out_path, bbox_inches='tight')
print(f"Saved figure as {save_name}")

plt.show()

### Layer-wise L2 distances

In [ ]:
architectures = ['alexnet', 'vgg16', 'resnet18', 'resnet34']
fs = 18
lw = 2
alpha_line = 0.9
alpha_ci = 0.15
ms = 8

def pretty_title(a):
    a = a.lower().replace('_','-')
    if a.startswith('alex'): return 'AlexNet'
    if a.startswith('vgg'): return 'VGG-' + ''.join(ch for ch in a if ch.isdigit())
    if a.startswith('resnet'): return 'ResNet-' + ''.join(ch for ch in a if ch.isdigit())
    return a

# Same fixed colors as cosine plot
arch_to_color = {
    'alexnet':  '#ff7f0e',
    'vgg16':    '#ffb300',
    'resnet18': '#008000', 
    'resnet34': '#1f77b4' 
}

def stack_model_scalar(model_data):
    """
    model_data: list over datasets; each item is list over layers of scalars
    returns: np.ndarray of shape (num_datasets, num_layers)
    """
    arrs = []
    for ds in model_data:
        a = np.asarray(ds, dtype=float)
        if a.ndim != 1:
            raise ValueError(f"Each dataset entry must be 1-D (num_layers,); got {a.shape}")
        arrs.append(a)
    return np.stack(arrs, axis=0) 

# L2 distance figure
fig, axs = plt.subplots(1, len(architectures), figsize=(22, 5.5))  

if len(architectures) == 1:
    axs = [axs]

for ax, arch in zip(axs, architectures):
    data = stack_model_scalar(L2_dist[arch])  
    mean_over_ds = data.mean(axis=0)
    std_over_ds  = data.std(axis=0, ddof=0)
    n_layers = mean_over_ds.shape[0]

    # relative depths
    try:
        rdepths = np.asarray(Depths[arch], dtype=float)
        assert rdepths.shape[0] == n_layers
        rdepths = rdepths / rdepths[-1]
    except Exception:
        rdepths = np.linspace(0.0, 1.0, n_layers)

    color = arch_to_color[arch]
    # mean line + shaded std
    ax.plot(rdepths, mean_over_ds, '-o', linewidth=lw, markersize=ms,
            alpha=alpha_line, color=color)
    ax.fill_between(rdepths, mean_over_ds - 2*std_over_ds, mean_over_ds + 2*std_over_ds,
                    color=color, alpha=alpha_ci)

    ax.grid(axis='y', alpha=0.3)
    ax.grid(axis='x', alpha=0.3)
    ax.set_xlim(-0.05, 1.05)
    ax.set_xticks(np.linspace(0, 1, 6))
    ax.tick_params(labelsize=fs - 2)
    ax.set_xlabel('Relative Depth', fontsize=fs)
    ax.set_title(pretty_title(arch), fontsize=fs+12, fontweight='bold', pad=17)

axs[0].set_ylabel('Average Representation Length', fontsize=fs)
fig.tight_layout()

# save figure
out_path = paths.cnn_figs_root / "avg_size_all_models.pdf"
fig.savefig(out_path, bbox_inches='tight')
print(f"Saved figure as {out_path}")
plt.show()


### Layer-wise Cosine Similarity

In [ ]:
architectures = ['alexnet', 'vgg16', 'resnet18', 'resnet34']
fs = 18
lw = 2
alpha_line = 0.9
alpha_ci = 0.15
ms = 8

def pretty_title(a):
    a = a.lower().replace('_','-')
    if a.startswith('alex'): return 'AlexNet'
    if a.startswith('vgg'): return 'VGG-' + ''.join(ch for ch in a if ch.isdigit())
    if a.startswith('resnet'): return 'ResNet-' + ''.join(ch for ch in a if ch.isdigit())
    return a

# colors 
arch_to_color = {
    'alexnet':  '#ff7f0e',
    'vgg16':    '#ffb300',
    'resnet18': '#008000',
    'resnet34': '#1f77b4' 
}

def stack_model_scalar(model_data):
    """
    model_data: list over datasets; each item is list over layers of scalars
    returns: np.ndarray of shape (num_datasets, num_layers)
    """
    arrs = []
    for ds in model_data:
        a = np.asarray(ds, dtype=float)
        if a.ndim != 1:
            raise ValueError(f"Each dataset entry must be 1-D (num_layers,); got {a.shape}")
        arrs.append(a)
    return np.stack(arrs, axis=0)  # (D, L)

# cosine similarity plot
fig, ax = plt.subplots(figsize=(12.5, 6.0))

for arch in list(reversed(architectures)):
    data = stack_model_scalar(cossim[arch])  
    mean_over_ds = data.mean(axis=0)
    std_over_ds  = data.std(axis=0, ddof=0)
    n_layers = mean_over_ds.shape[0]

    # relative depths
    try:
        rdepths = np.asarray(Depths[arch], dtype=float)
        assert rdepths.shape[0] == n_layers
        rdepths = rdepths / rdepths[-1]
    except Exception:
        rdepths = np.linspace(0.0, 1.0, n_layers)

    color = arch_to_color[arch]
    ax.plot(rdepths, mean_over_ds, '-o', linewidth=lw, markersize=ms,
            alpha=alpha_line, color=color, label=pretty_title(arch))
    ax.fill_between(rdepths, mean_over_ds - 2*std_over_ds, mean_over_ds + 2*std_over_ds,
                    color=color, alpha=alpha_ci)

ax.grid(axis='both', alpha=0.3)
ax.set_xlim(-0.05, 1.05)
ax.set_xticks(np.linspace(0, 1, 6))
ax.tick_params(labelsize=fs - 2)
ax.set_xlabel('Relative Depth', fontsize=fs)
ax.set_ylabel('Average Cosine Similarity', fontsize=fs)

# Legend on the right
legend_handles = [
    Line2D([0], [0], marker='o', linestyle='-', linewidth=lw, markersize=ms,
           color=arch_to_color[a], label=pretty_title(a))
    for a in architectures
]
plt.subplots_adjust(right=0.82)
fig.legend(handles=legend_handles, labels=[pretty_title(a) for a in architectures],
           fontsize=fs - 2, loc='center left', bbox_to_anchor=(0.845, 0.5), frameon=False)
fig.tight_layout(rect=[0, 0, 0.85, 1])

# save figure
out_path = paths.cnn_figs_root / "avg_cossim_all_models.pdf"
fig.savefig(out_path, bbox_inches='tight')
print(f"Saved figure as {out_path}")
plt.show()

### ID vs. Entropy Analysis

In [ ]:
lw = 2.5
alpha = 0.6
ms = 10
fs = 20
c_id  = '#0000ff'  
c_ent = '#ffb300' 

def pretty_title(a):
    a = a.lower().replace('_','-')
    if a.startswith('alex'): return 'AlexNet'
    if a.startswith('vgg'): return 'VGG-' + ''.join(ch for ch in a if ch.isdigit())
    if a.startswith('resnet'): return 'ResNet-' + ''.join(ch for ch in a if ch.isdigit())
    return a

def plot_panel(ax1, arch):
    rdepths = Depths[arch] / Depths[arch][-1]
    # ID (left)
    id_mean = np.mean(ID[arch][0:-1, :], 0)
    id_std  = np.std(ID[arch][0:-1, :], 0)
    ax1.plot(rdepths, id_mean, '-', color=c_id, lw=lw,
             marker='o', ms=ms, mfc=c_id, mec='white', mew=1.0, label='Intrinsic Dimension')
    ax1.errorbar(rdepths, id_mean, yerr=id_std, fmt='none',
                 ecolor=c_id, elinewidth=lw, alpha=alpha)

    ax1.set_xlabel('Relative Depth', fontsize=fs+2)
    ax1.set_ylabel('ID', fontsize=fs+4)
    ax1.tick_params(axis='y', labelsize=fs, top=False, right=False)
    ax1.tick_params(axis='x', labelsize=fs, top=False)
    ax1.set_xlim(-0.1, 1.1)
    ax1.set_xticks(np.arange(0, 1.01, 0.2))
    ax1.set_xticklabels(np.round(np.arange(0, 1.01, 0.2), 1), fontsize=fs)
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)

    # Entropy (right)
    ax2 = ax1.twinx()
    entropy_mean = np.mean(Entropy[arch][0:-1, :], 0)
    entropy_std  = np.std(Entropy[arch][0:-1, :], 0)
    ax2.plot(rdepths, entropy_mean, '-', color=c_ent, lw=lw,
             marker='o', ms=ms, mfc=c_ent, mec='white', mew=1.0, label='von Neumann Entropy')
    ax2.errorbar(rdepths, entropy_mean, yerr=entropy_std, fmt='none',
                 ecolor=c_ent, elinewidth=lw, alpha=alpha)
    ax2.set_ylabel('Entropy', fontsize=fs+4)
    ax2.tick_params(axis='y', labelsize=fs, top=False)
    ax2.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    ax2.spines['top'].set_visible(True)
    ax2.spines['right'].set_visible(True)
    ax2.set_ylim(bottom=None, top=6.15)

    ax1.set_title(pretty_title(arch), fontsize=fs+12, fontweight='bold', pad=17)
    l1, lab1 = ax1.get_legend_handles_labels()
    l2, lab2 = ax2.get_legend_handles_labels()
    return l1+l2, lab1+lab2

# creat 1x4 row 
archs_row = ['alexnet', 'vgg16', 'resnet18', 'resnet34'] 
fig, axes = plt.subplots(1, 4, figsize=(24, 6), sharex=False)
all_h, all_l = [], []
for ax, a in zip(axes, archs_row):
    h, l = plot_panel(ax, a)
    all_h, all_l = h, l 

# only show left y-label on the first subplot
for ax in axes[1:]:
    ax.set_ylabel("")       

for ax in ax.figure.axes[4:7]:
    ax.set_ylabel("")       

fig.legend(all_h, all_l,
           loc='lower center',       
           ncol=2,                       
           fontsize=fs+8,
           frameon=False,
           bbox_to_anchor=(0.5, -0.1),  
           handlelength=1.8, columnspacing=1.5, handletextpad=0.5)

plt.tight_layout(rect=[0, 0.05, 1, 1], w_pad=2.2)
# save figure
out_path = paths.cnn_figs_root / "id_vs_entropy.pdf"
fig.savefig(out_path, bbox_inches='tight')
print(f"Saved figure as {out_path}")
plt.show()


### ID vs. Ambient Space Dimension

In [ ]:
lw = 2.5
alpha = 0.6
ms = 10
fs = 20
c_id  = '#0000ff'  
c_ent = '#008000'  

def pretty_title(a):
    a = a.lower().replace('_','-')
    if a.startswith('alex'): return 'AlexNet'
    if a.startswith('vgg'): return 'VGG-' + ''.join(ch for ch in a if ch.isdigit())
    if a.startswith('resnet'): return 'ResNet-' + ''.join(ch for ch in a if ch.isdigit())
    return a

def plot_panel(ax1, arch):
    rdepths = Depths[arch] / Depths[arch][-1]
    # ID (left)
    id_mean = np.mean(ID[arch][0:-1, :], 0)
    id_std  = np.std(ID[arch][0:-1, :], 0)
    ax1.plot(rdepths, id_mean, '-', color=c_id, lw=lw,
             marker='o', ms=ms, mfc=c_id, mec='white', mew=1.0, label='Intrinsic Dimension')
    ax1.errorbar(rdepths, id_mean, yerr=id_std, fmt='none',
                 ecolor=c_id, elinewidth=lw, alpha=alpha)

    ax1.set_xlabel('Relative Depth', fontsize=fs+2)
    ax1.set_ylabel('ID', fontsize=fs+4)
    ax1.tick_params(axis='y', labelsize=fs, top=False, right=False)
    ax1.tick_params(axis='x', labelsize=fs, top=False)
    ax1.set_xlim(-0.1, 1.1)
    ax1.set_xticks(np.arange(0, 1.01, 0.2))
    ax1.set_xticklabels(np.round(np.arange(0, 1.01, 0.2), 1), fontsize=fs)
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)

    # Ambient space (right)
    ax2 = ax1.twinx()
    n_emdsims = int(ambdims[arch].shape[0] / ID[arch].shape[0])
    emd_dim = ambdims[arch][0:n_emdsims]
    ax2.plot(rdepths, emd_dim, '-', color=c_ent, lw=lw,
             marker='o', ms=ms, mfc=c_ent, mec='white', mew=1.0, label='Ambient Space Dimension')
    ax2.set_ylabel('Ambient Space Dimension', fontsize=fs+4)
    ax2.tick_params(axis='y', labelsize=fs, top=False)
    ax2.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    ax2.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{int(x/1000)}k")
)
    ax2.spines['top'].set_visible(True)
    ax2.spines['right'].set_visible(True)

    ax1.set_title(pretty_title(arch), fontsize=fs+12, fontweight='bold', pad=17)
    l1, lab1 = ax1.get_legend_handles_labels()
    l2, lab2 = ax2.get_legend_handles_labels()
    return l1+l2, lab1+lab2

# create 1x4 row
archs_row = ['alexnet', 'vgg16', 'resnet18', 'resnet34'] 
fig, axes = plt.subplots(1, 4, figsize=(24, 6), sharex=False)
all_h, all_l = [], []
for ax, a in zip(axes, archs_row):
    h, l = plot_panel(ax, a)
    all_h, all_l = h, l 

# only show left y-label on the first subplot
for ax in axes[1:]:
    ax.set_ylabel("")       

for ax in ax.figure.axes[4:7]:
    ax.set_ylabel("")       

fig.legend(all_h, all_l,
           loc='lower center',           
           ncol=2,                       
           fontsize=fs+8,
           frameon=False,
           bbox_to_anchor=(0.5, -0.1), 
           handlelength=1.8, columnspacing=1.5, handletextpad=0.5)

plt.tight_layout(rect=[0, 0.05, 1, 1], w_pad=2.2)
# save figure
out_path = paths.cnn_figs_root / "id_vs_ambient_dim.pdf"
fig.savefig(out_path, bbox_inches='tight')
print(f"Saved figure as {out_path}")
plt.show()


### Plot IDs over all models

In [ ]:
archs = ['alexnet', 'vgg11', 'vgg13', 'vgg16','vgg19',
                    'vgg11_bn', 'vgg13_bn', 'vgg16_bn','vgg19_bn',
                    'resnet18', 'resnet34', 'resnet50', 'resnet101', 'resnet152']

In [ ]:
def get_color(arch: str):
    a = arch.lower()
    if 'alex' in a:    return '#ff7f0e'  
    if 'bn' in a:      return '#008000'  
    if 'vgg' in a:     return '#ffb300'  
    if 'res' in a:     return '#7f7f7f'  
    return '#7f7f7f'   


def pretty_label(arch: str):
    s = arch.lower().replace('_', '-')
    num = re.findall(r'\d+', s)
    num = num[0] if num else ''
    has_bn = 'bn' in s
    if s.startswith('alex'): return 'alexnet'
    if s.startswith('vgg'):
        base = f'vgg{num}' if num else 'vgg'
        return f'{base}_bn' if has_bn else base
    if s.startswith('res'):  return f'resnet{num}' if num else 'resnet'
    return arch

def model_number(arch):
    m = re.search(r'(\d+)', arch)
    return int(m.group(1)) if m else 0

# figure style 
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': False,
    'axes.edgecolor': 'black',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'legend.frameon': False
})

lw, ms, fs = 2.5, 9, 22
msexp = [1,
         1,1.1,1.2,1.3,
         1,1.1,1.2,1.3,
         1,1.1,1.2,1.3,1.4]

fig, ax = plt.subplots(figsize=(10, 6))

for i, arch in enumerate(archs):
    rdepths = Depths[arch] / Depths[arch][-1]
    y = np.mean(ID[arch][0:-1, :], 0)

    c = get_color(arch)
    mface = c
    ax.plot(
        rdepths, y,
        '-', color=c, lw=lw,
        marker='o', ms=ms * msexp[i],
        mfc=mface, mec='white', mew=1.5,
        label=pretty_label(arch)
    )

ax.set_xlabel('Relative Depth', fontsize=fs)
ax.set_ylabel('Estimated Intrinsic Dimension', fontsize=fs)
ax.set_xlim(-0.1, 1.1)
ax.set_ylim(0, 160)
ax.tick_params(labelsize=fs, top=False, right=False)

# legend
handles, labels = ax.get_legend_handles_labels()

def sort_key(lbl):
    s = lbl.lower()
    if s.startswith('alex'): grp = 0
    elif s.startswith('vgg') and 'bn' not in s: grp = 1
    elif s.startswith('vgg') and 'bn' in s: grp = 2
    elif s.startswith('res'): grp = 3
    else: grp = 9
    nums = re.findall(r'\d+', s)
    num = int(nums[0]) if nums else 0
    return (grp, num)

order = sorted(range(len(labels)), key=lambda i: sort_key(labels[i]))
handles = [handles[i] for i in order]
labels  = [labels[i] for i in order]

resnet18_idx = [i for i, lbl in enumerate(labels) if lbl.lower() == "resnet18"]
if resnet18_idx:
    idx = resnet18_idx[0]
    resnet18_handle = handles.pop(idx)
    resnet18_label  = labels.pop(idx)
else:
    resnet18_handle, resnet18_label = None, None

# Insert a blank entry at the end of column 2 (index 9 for 3x5 legend)
blank = mlines.Line2D([], [], linestyle='None', marker=None, alpha=0)
handles.insert(9, blank)
labels.insert(9, " ")

# Insert resnet18 at the top of column 3 (index 10)
if resnet18_handle is not None:
    handles.insert(10, resnet18_handle)
    labels.insert(10, resnet18_label)

proxy_labels = [
    'AlexNet',
    'VGG-11',
    'VGG-13',
    'VGG-16',
    'VGG-19',
    'VGG-11 (BN)',
    'VGG-13 (BN)',
    'VGG-16 (BN)',
    'VGG-19 (BN)',
    '',
    'ResNet-18',
    'ResNet-34',
    'ResNet-50',
    'ResNet-101',
    'ResNet-152'
]

ax.legend(
    handles, proxy_labels,
    ncol=3, fontsize=15, handlelength=1.8, columnspacing=1.6,
    loc='upper center', bbox_to_anchor=(0.5, 1.1)
)

fig.tight_layout()
plt.show()
out_path = paths.cnn_figs_root / "id_patterns.pdf"
fig.savefig(out_path, bbox_inches='tight')
print(f"Saved figure as {out_path}")
